In [9]:
import pandas as pd 
import json 
import importlib
from pathlib import Path
from utils import vis_utils, rxn_vis, prolif_utils

importlib.reload(vis_utils)


<module 'utils.vis_utils' from '/mnt/home/jfromer/collabs/docking/visualization/utils/vis_utils.py'>

## Results of SynFormer-guided MOR docking screen

In [10]:
buffer_order = [
    'Objective', 
    'Oracle call',
    'Docking score',
    'Best conformer',
    'Charge',
    'Max consecutive rotatable bonds', 
    'Unbound H bond donors',
    'Interacts with water',
    'Interacts with ASP',
    'Synthesis'
]

def load_df(mol_buffer_file, max_entries: int = 100000, outfile: str = 'data/preinitialized.csv'): 
    with open(mol_buffer_file, 'r') as f: 
        mol_buffer = json.load(f)

    mol_buffer = {
        smi: val for smi, val in mol_buffer.items() 
        if len(val) >= len(buffer_order)
    }

    df = pd.DataFrame({'SMILES': list(mol_buffer.keys())})
    for i, column_name in enumerate(buffer_order): 
        df[column_name] = [
            val[i] for val in mol_buffer.values()
        ]

    df.sort_values(by='Objective', inplace=True, ascending=False)
    df['Rank'] = [i+1 for i in range(len(df))]

    df = vis_utils.initialize_df(df.loc[:max_entries].copy())

    # save 
    df.to_csv(outfile, index=False)
    return df

In [11]:
df_file = 'data/preinitialized.csv'
mol_buffer_file = 'data/results_mordocking_10_19.json'

if Path(df_file).exists(): 
    df = pd.read_csv(df_file)
else: 
    df = load_df(mol_buffer_file)

Getting synthesis: 100%|██████████| 100046/100046 [00:00<00:00, 1400630.64it/s]


In [12]:
importlib.reload(rxn_vis)
importlib.reload(prolif_utils)
importlib.reload(vis_utils)

visualizer = vis_utils.MoleculeGridSelectorWithFilters(df, n_rows=2, n_cols=5)
visualizer.display()

/home/jfromer/miniforge3/envs/medchem/lib/python3.9/site-packages/MDAnalysis/converters/RDKit.py:473: UserWarning: No `bonds` attribute in this AtomGroup. Guessing bonds based on atoms coordinates
  warnings.warn(
